In this notebook we prepare data to be plotted in the actual figure generation notebook.

The routines used here are from `20251029_repeat_beam_aim_investigate.ipynb`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from infotaxis import one_target, hex_ops

In [2]:
# Path to save figs and data for figs
path_fig = Path("/Users/wujung/code_git/infotaxis/figs_paper")
path_data = Path("/Users/wujung/code_git/infotaxis/notebooks_fig")

## Set params

In [3]:
cr_all = list(range(5, 11))
br_all = [1, 2]
pm = 0.001

pfa_precision_all = [1e-3, 1e-4, 1e-5]
pfa_start = np.arange(0, 0.021, 0.001)

In [4]:
path_csv = path_data / f"20251221_h_est_pm{pm:.0e}"
if not path_csv.exists():
    path_csv.mkdir(parents=True, exist_ok=True)

## Compute the pfa crossover points

In [5]:
def get_cross_over_idx(
    pfa_array, cr, br, pm, center_cube=(0, 0, 0), neighbor_cube = (1, 0, -1)
):

    # Compute h_est for repeating or moving beam aim (h_est_c, h_est_n)
    h_est_c_all = []
    h_est_n_all = []

    for pfa in pfa_array:
        print("--------------------------------")
        print(f"pfa={pfa}")

        search_dict = None
        param_echo = {
            str(br): {
                "pm_const": pm,
                "pfa_const": pfa,
            },
        }

        oth = one_target.OneTargetHex(
            search_rule="infotaxis",
            canvas_radius=cr,
            beam_radius=[br],
            target_cube=(-2, 3, -1),
            aim_start_cube=center_cube, # assign first beam aim
            param_animal= param_echo,
            param_echo= param_echo,
            search_dict=search_dict
        )

        # Assign echo outcome
        oth.echo_value = True # receive echo
        oth.echo_type = True  # correct return

        # Update map after receiving echo
        oth.update_X1()
        oth.get_est_ph()

        # Get sequence index of the center and neighbor grids
        k_center = hex_ops.axial_to_k(hex_ops.cube_to_axial(center_cube), oth.canvas_axial)
        k_neighbor = hex_ops.axial_to_k(hex_ops.cube_to_axial(neighbor_cube), oth.canvas_axial)

        # Find flat index of where k_center and k_neighbor in oth.h_est[br]
        seq_idx_center = np.where(oth.k_canvas==k_center)[0][0]
        seq_idx_neighbor = np.where(oth.k_canvas==k_neighbor)[0][0]

        # Get expected entropy for aiming at the center and neighbor grids
        h_est_c = oth.h_est[br][seq_idx_center]
        h_est_n = oth.h_est[br][seq_idx_neighbor]

        # Store results
        h_est_c_all.append(h_est_c)
        h_est_n_all.append(h_est_n)

    h_est_c_all = np.array(h_est_c_all)
    h_est_n_all = np.array(h_est_n_all)

    # Get cross over start point
    # -1 because cross over happens AFTER h_est_n_all > h_est_c_all
    idx_fine_start = (h_est_n_all < h_est_c_all).sum() -1

    return idx_fine_start, h_est_c_all, h_est_n_all

In [6]:
result_list = []

for cr in cr_all:
    for br in br_all:

        # Get crossover pfa value
        for seq, pfa_p in enumerate(pfa_precision_all):
            print("=========================================================================")
            print(f"cr={cr}, br={br}, pfa precision: {pfa_p}")

            if seq == 0:
                pfa_all = pfa_start
                # idx_fine_x = 0 # initialize, no meaning
            else:
                pfa_all = np.arange(pfa_all[idx_x], pfa_all[idx_x+1]+pfa_p, pfa_p)

            print("pfa_all under consideration:")
            print(pfa_all)
            print("=========================================================================")
            idx_x, h_est_c, h_est_n = get_cross_over_idx(pfa_array=pfa_all, cr=cr, br=br, pm=pm)

            # Save repeat the same grid (h_est_c) and move to neighbor (h_est_n)
            df = pd.DataFrame(
                [pfa_all, h_est_c, h_est_n],
                index=["pfa", "h_est_c", "h_est_n"]
            ).T
            df["cr"] = cr
            df["br"] = br
            fname = f"cr{cr}_br{br}_pm{pm:.0e}_pfaPrecision{pfa_p:.0e}.csv"
            df.to_csv(path_csv / fname)

        pfa_x_value = pfa_all[idx_x]
        h_est_c_x_value = h_est_c[idx_x]
        h_est_n_x_value = h_est_n[idx_x]

        # Store values
        result_list.append([cr, br, pm, pfa_x_value, h_est_c_x_value, h_est_n_x_value])

cr=5, br=1, pfa precision: 0.001
pfa_all under consideration:
[0.    0.001 0.002 0.003 0.004 0.005 0.006 0.007 0.008 0.009 0.01  0.011
 0.012 0.013 0.014 0.015 0.016 0.017 0.018 0.019 0.02 ]
--------------------------------
pfa=0.0
Initial beam aim is given, but not initial beam radius.
Set initial beam radius to the largest of beam radius choices: 1
--------------------------------
pfa=0.001
Initial beam aim is given, but not initial beam radius.
Set initial beam radius to the largest of beam radius choices: 1
--------------------------------
pfa=0.002
Initial beam aim is given, but not initial beam radius.
Set initial beam radius to the largest of beam radius choices: 1
--------------------------------
pfa=0.003
Initial beam aim is given, but not initial beam radius.
Set initial beam radius to the largest of beam radius choices: 1
--------------------------------
pfa=0.004
Initial beam aim is given, but not initial beam radius.
Set initial beam radius to the largest of beam radius ch

In [7]:
df_h = pd.DataFrame(
    result_list,
    columns = ["cr", "br", "pm", "pfa_crossover_value", "h_est_c_crossover_value", "h_est_n_crossover_value"]
)
df_h

,cr,br,pm,pfa_crossover_value,h_est_c_crossover_value,h_est_n_crossover_value
0,5,1,0.001,0.00629,4.141250,4.141213
1,5,2,0.001,0.00908,5.269737,5.269725
2,6,1,0.001,0.00445,4.308180,4.307567
3,6,2,0.001,0.00608,5.440888,5.440752
4,7,1,0.001,0.00333,4.455708,4.455043
5,7,2,0.001,0.00442,5.595892,5.595685
6,8,1,0.001,0.00259,4.586459,4.585747
7,8,2,0.001,0.00338,5.735617,5.735231
8,9,1,0.001,0.00207,4.701286,4.699974
9,9,2,0.001,0.00268,5.863189,5.862711


In [8]:
df_h.to_csv(path_data / f"20251221_pfa_crossover_pm{pm:.0e}.csv")